# Embeddings Interface Reference

Developer-facing statements defined in `langchain_core.embeddings.embeddings`.

# `Embeddings: ABC`

Abstract interface for text embedding models.

It maps document text and query text to vectors. Document embedding returns one vector per input string, while query embedding returns a single vector. A concrete subclass must implement `embed_documents` and `embed_query`.

## Required subclass hooks

### `embed_documents`

Embeds a list of search documents.

```python
embed_documents(
    self,
    texts: list[str], # Texts to embed
) -> list[list[float]] # One embedding vector for each input text
```

### `embed_query`

Embeds a query string.

```python
embed_query(
    self,
    text: str, # Query text to embed
) -> list[float] # Query embedding vector
```

## Methods

### `aembed_documents`

Asynchronously embeds a list of search documents.

```python
async aembed_documents(
    self,
    texts: list[str], # Texts to embed
) -> list[list[float]] # One embedding vector for each input text
```

The default implementation calls `embed_documents` through `run_in_executor`. Subclasses may override it with a native asynchronous implementation.

### `aembed_query`

Asynchronously embeds a query string.

```python
async aembed_query(
    self,
    text: str, # Query text to embed
) -> list[float] # Query embedding vector
```

The default implementation calls `embed_query` through `run_in_executor`. Subclasses may override it with a native asynchronous implementation.

## Behaviour

Document and query embeddings may use identical logic, but the interface allows implementations to treat them differently.

In [ ]:
#%pip install -U langchain-core#Install LangChain Core if it is not already installed

import math#Import math for calculating vector magnitude
import re#Import regular expressions for extracting words
from langchain_core.embeddings import Embeddings#Import the abstract Embeddings interface


class KeywordEmbeddings(Embeddings):#Create a custom keyword-based embedding model

    def __init__(self, vocabulary: list[str]) -> None:#Initialize the model with a fixed vocabulary
        self.vocabulary = [word.lower() for word in vocabulary]#Store every vocabulary word in lowercase

    def _create_vector(self, text: str) -> list[float]:#Convert one text value into a numeric vector
        words = re.findall(r"\b\w+\b", text.lower())#Extract lowercase words from the text
        vector = [float(words.count(word)) for word in self.vocabulary]#Count every vocabulary word in the text
        return vector#Return the generated vector

    def embed_documents(self, texts: list[str]) -> list[list[float]]:#Embed multiple document strings
        return [self._create_vector(text) for text in texts]#Create one vector for every document

    def embed_query(self, text: str) -> list[float]:#Embed one search query
        return self._create_vector(text)#Create and return the query vector


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:#Calculate similarity between two vectors
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))#Calculate the vector dot product
    magnitude_a = math.sqrt(sum(value ** 2 for value in vector_a))#Calculate the magnitude of the first vector
    magnitude_b = math.sqrt(sum(value ** 2 for value in vector_b))#Calculate the magnitude of the second vector

    if magnitude_a == 0 or magnitude_b == 0:#Check whether either vector contains only zeros
        return 0.0#Return zero similarity when comparison is impossible

    return dot_product / (magnitude_a * magnitude_b)#Return the cosine-similarity score


vocabulary = [#Create the words represented by the embedding dimensions
    "password",#Represent password-related content
    "account",#Represent account-related content
    "refund",#Represent refund-related content
    "payment",#Represent payment-related content
    "delivery",#Represent delivery-related content
    "order",#Represent order-related content
]#Finish the vocabulary list

faq_documents = [#Create sample FAQ documents
    "You can reset your account password from the account settings page.",#Create the password FAQ
    "Refunds are returned to the original payment method within seven days.",#Create the refund FAQ
    "You can track your delivery using the order tracking page.",#Create the delivery FAQ
]#Finish the FAQ document list

embedding_model = KeywordEmbeddings(vocabulary)#Create the custom embedding model
document_vectors = embedding_model.embed_documents(faq_documents)#Convert all FAQ documents into vectors

user_query = "How do I change my account password?"#Define the user's search query
query_vector = embedding_model.embed_query(user_query)#Convert the query into a vector

scored_documents = []#Create an empty list for documents and similarity scores

for document, document_vector in zip(faq_documents, document_vectors):#Compare the query with every FAQ document
    score = cosine_similarity(query_vector, document_vector)#Calculate the query-document similarity
    scored_documents.append((score, document))#Store the score with its corresponding document

scored_documents.sort(key=lambda item: item[0], reverse=True)#Sort documents from highest to lowest similarity

best_score, best_document = scored_documents[0]#Select the most relevant document

print(f"Query: {user_query}")#Display the user's query
print(f"Query vector: {query_vector}")#Display the generated query vector
print("\nDocument rankings:")#Display the ranking heading

for score, document in scored_documents:#Process every ranked document
    print(f"{score:.3f} -> {document}")#Display its similarity score and content

print("\nBest matching answer:")#Display the final-answer heading
print(best_document)#Display the most relevant FAQ document